# Chaînes-04 — L'indiscernabilité et le signe spin-statistique

Série [Chaînes](README.md), grain 4 — distillation du chapitre 43 du compendium Chain Theory
(Wissner-Gross, 2026, preprint non relu) : *Indistinguishability and the spin-statistics sign*.
Le fil : après l'acte de la mesure (Chaînes-03), ce que la construction dit des corps qu'on ne
peut pas distinguer — deux copies d'un site lues comme un seul qudit de dimension $d^2$, le swap
lavé par le twirl du groupe d'identification, et le signe de l'échange lu à trois faces.

Patron de la série : **Mesuré** (rejoué par le code de ce carnet) · **Cité** (toute la chaîne
dérivative de la source, jamais rejouée ici) · **Ouvert** (ce que la source ne prouve pas,
listé ligne à ligne, jamais maquillé en résultat établi).

In [1]:
import numpy as np

## Le cadre : lire deux copies comme un qudit

Le théorème 43.1 est **conditionnel** — `Standing: a conditional theorem` — posé sur l'axiome et
les chapitres nommés en prémices. Sous la lecture qui prend les deux copies d'un site de
dimension $d$ comme **un seul qudit de dimension $d^2$** :

- la configuration debout à deux copies se construit aux **dix-huit objets** de la construction
  (treize corps, cinq porteurs) — *census, cité plus bas* ;
- le swap des copies se réalise comme une **identification de base** à chaque objet, puis se lave
  par le twirl du groupe d'identification — *la construction du twirl appartient à la source* ;
- le signe de l'échange se lit à **trois faces** : la face amplitude, la face permutation, et le
  cycle impair de la configuration debout.

Ce carnet rejoue les trois faces au cas élémentaire $d = 2$ (le type deux-qubits), plus les
identités arithmétiques que la source invite explicitement le lecteur à reprendre à la main.

In [2]:
# Corps du carnet : les deux machines elementaires du chapitre.

# 1) Le signe de la permutation des etiquettes produit : l'echange de deux sous-systemes de
#    dimension d est une permutation des d^2 etiquettes produit (i, j) -> (j, i).
def signe_permutation_etiquettes(d):
    """Signe, nombre de transpositions et nombre d'etiquettes de la permutation (i,j) -> (j,i)."""
    n = d * d
    perm = [(k % d) * d + k // d for k in range(n)]
    vus = [False] * n
    signe, nb_transp = 1, 0
    for k in range(n):
        if not vus[k]:
            longueur, c = 0, k
            while not vus[c]:
                vus[c] = True
                c = perm[c]
                longueur += 1
            nb_transp += longueur - 1
            if longueur % 2 == 0:
                signe = -signe
    return signe, nb_transp, n

# 2) L'echange des copies sur une paire doublee : v (x) v sur (a1 b1)(a2 b2),
#    l'echange des copies transpose les deux blocs.
def paire_double_fixee(v):
    """La paire doublee v (x) v est-elle fixee par l'echange des copies, et avec quelle amplitude ?"""
    ket = np.kron(v, v)
    echange = ket.reshape(len(v), len(v)).T.reshape(-1)
    return np.allclose(echange, ket), float(np.vdot(ket, echange))

# Essai des deux machines au cas elementaire d = 2.
print("essai signe_permutation_etiquettes(2) :", signe_permutation_etiquettes(2))
v_essai = np.array([1.0, 0.0, 0.0, 1.0]) / np.sqrt(2.0)
print("essai paire_double_fixee(|Phi+>)      :", paire_double_fixee(v_essai))

essai signe_permutation_etiquettes(2) : (-1, 1, 4)
essai paire_double_fixee(|Phi+>)      : (True, 0.9999999999999996)


### Lecture du résultat

Les deux machines sont volontairement minuscules : la première décompose la permutation des
étiquettes en cycles et en tire le signe — c'est la **face permutation** de l'échange ; la
deuxière applique l'échange des copies à une paire doublée $v \otimes v$ — c'est la **face
amplitude**. Tout le chapitre vit dans l'écart entre ces deux faces.

## Équation 61 — la paire à deux copies au type deux-qubits

La partie exacte de la dérivation, au type deux-qubits : deux copies d'un site de dimension 2,
chacune portant la paire à deux copies — l'état $\lvert \Phi^+ \rangle$ de Schmidt rang 2 —
donnent la configuration à deux copies de dimension de registre **16**. Le swap des copies fixe
le ket de paire à deux copies avec le **signe d'amplitude plus un** et le **signe de permutation
moins un**, au **rang de Schmidt quatre** :

$$r = 2, \quad D = 4, \quad \dim R = 16, \quad \sigma_{\text{amp}} = +1, \quad \sigma_{\text{perm}} = -1. \qquad (61)$$

**Mesuré** : les quatre quantités de la ligne, au sens machine. Précision de bipartition : le
rang de Schmidt quatre se prend à travers la coupure **des sites** doublée — $(A_1 A_2)$ contre
$(B_1 B_2)$ — où le rang vaut $r^2 = 4$ ; à travers la coupure **des copies**, l'état est un
produit ($\lvert\Phi^+\rangle \otimes \lvert\Phi^+\rangle$) et le rang vaut un.

In [3]:
# La paire a deux copies : |Phi+> sur (A1,B1) tenseur |Phi+> sur (A2,B2).
ket_phi = np.array([1.0, 0.0, 0.0, 1.0]) / np.sqrt(2.0)
ket_omega = np.kron(ket_phi, ket_phi)          # registre (A1,B1,A2,B2), dim 16

# r = rang de Schmidt de |Phi+> a travers A|B ; D = support double = 2r ; dim R = 16
r = int(round(np.linalg.matrix_rank(ket_phi.reshape(2, 2))))
D_double = 2 * r
dim_R = ket_omega.size

# Rang de Schmidt de |Omega> : ACROSS les sites doubles (A1,A2) | (B1,B2) -- rang r^2 = 4.
# (A travers la coupure des copies (A1,B1)|(A2,B2), l'etat est un produit, de rang 1.)
M_sites = ket_omega.reshape(2, 2, 2, 2).transpose(0, 2, 1, 3).reshape(4, 4)
sing_sites = np.linalg.svd(M_sites, compute_uv=False)
rang_schmidt_sites = int((sing_sites > 1e-12).sum())
M_copies = ket_omega.reshape(4, 4)
rang_copies = int((np.linalg.svd(M_copies, compute_uv=False) > 1e-12).sum())

# sigma_amp : l'echange des copies echange les blocs (A1,B1) <-> (A2,B2)
echange = ket_omega.reshape(4, 4).T.reshape(-1)
sigma_amp = float(np.vdot(ket_omega, echange))
fixe_exactement = np.allclose(echange, ket_omega)

# sigma_perm : signe de la permutation des etiquettes produit a d = 2
sigma_perm, nb_transp, n_etiquettes = signe_permutation_etiquettes(2)

print(f"r = {r}, D = {D_double}, dim R = {dim_R}")
print(f"rang de Schmidt a travers les sites doubles (A1A2)|(B1B2) : {rang_schmidt_sites}")
print(f"rang a travers la coupure des copies (produit) : {rang_copies}")
print(f"sigma_amp = {sigma_amp:+.15f}  (ket fixe exactement : {fixe_exactement})")
print(f"sigma_perm = {sigma_perm:+d}  ({nb_transp} transposition, {n_etiquettes} etiquettes)")

r = 2, D = 4, dim R = 16
rang de Schmidt a travers les sites doubles (A1A2)|(B1B2) : 4
rang a travers la coupure des copies (produit) : 1
sigma_amp = +1.000000000000000  (ket fixe exactement : True)
sigma_perm = -1  (1 transposition, 4 etiquettes)


### Lecture du résultat

La ligne (61) est reproduite exactement : $r = 2$, $D = 4$, $\dim R = 16$, et le rang de Schmidt
$4 = r^2$ à travers les sites doublés $(A_1 A_2)|(B_1 B_2)$ — deux copies d'un état de rang 2
doublent leur rang à travers la coupure d'origine. À travers la coupure **des copies**, le même
état est un produit, de rang un : la bipartition fait le rang, et la source mesure à celle des
sites. $\sigma_{\text{amp}} = +1$ à $10^{-15}$ — le ket de paire doublé est un vecteur propre
exact de l'échange des copies — et $\sigma_{\text{perm}} = -1$ : la même transposition, lue
cette fois comme permutation des quatre étiquettes produit, porte le signe moins. **Le signe
dépend de la face à laquelle on le lit** : c'est la phrase centrale du chapitre, et le carnet la
tient pour ce qu'elle est — une coexistence mesurée, pas une contradiction résolue.

## La face permutation à dimension quelconque

La source cite la correspondance [33] : l'échange de deux sous-systèmes de dimension $d$ est une
permutation des $d^2$ étiquettes produit avec $d(d-1)/2$ transpositions. Le tableau du chapitre
énumère les transpositions à chaque objet ; ici on rejoue la **formule** pour $d = 2, 3, 4$ :
nombre de transpositions $d(d-1)/2$, signe $(-1)^{d(d-1)/2}$, et le nombre d'étiquettes fixes
(les $d$ étiquettes diagonales $(i,i)$).

In [4]:
for d in (2, 3, 4):
    signe, nb_transp, n = signe_permutation_etiquettes(d)
    formule_transp = d * (d - 1) // 2
    etiquettes_fixes = d          # les (i, i) : la diagonale
    print(f"d = {d} : {n} etiquettes, {nb_transp} transpositions "
          f"(formule d(d-1)/2 = {formule_transp}), signe {signe:+d}, "
          f"{etiquettes_fixes} etiquettes fixes")
    assert nb_transp == formule_transp

d = 2 : 4 etiquettes, 1 transpositions (formule d(d-1)/2 = 1), signe -1, 2 etiquettes fixes
d = 3 : 9 etiquettes, 3 transpositions (formule d(d-1)/2 = 3), signe -1, 3 etiquettes fixes
d = 4 : 16 etiquettes, 6 transpositions (formule d(d-1)/2 = 6), signe +1, 4 etiquettes fixes


### Lecture du résultat

Le compte machine rejoint la formule $d(d-1)/2$ aux trois dimensions — la correspondance [33]
est bien une identité de théorie des groupes, indépendante de la construction. À noter : le
signe alterne $-1, -1, +1$ — la permutation n'est impaire qu'à $d \equiv 2, 3 \pmod 4$ ; à
$d = 4$ (le hub du chapitre) elle est déjà paire. La source ne commente pas cette alternance :
elle n'est pas mesurée ici au-delà des trois lignes du tableau.

## Équation 62 — les cinq lectures de la configuration twirlée

L'ensemble fixé à un type est le commutant du groupe d'identification sur le registre à deux
copies ; la **lecture** d'un opérateur $M$ à la configuration est

$$\langle M \rangle = \frac{\mathrm{tr}(M \, \Pi)}{16},$$

où $\Pi$ est le projecteur de paire à deux copies (rang un sur le support). La base du commutant
est $\{1, \Pi, O_1, O_2, O_3\}$ — les tables du twirl listées par la source — et les cinq
lectures valent :

$$\langle 1 \rangle = \tfrac{1}{16}, \quad \langle \Pi \rangle = \tfrac{1}{16}, \quad
\langle O_1 \rangle = \tfrac{1}{16}, \quad \langle O_2 \rangle = \tfrac{D-1}{16} = \tfrac{3}{16}, \quad
\langle O_3 \rangle = 0. \qquad (62)$$

La source invite le lecteur à reprendre **à la main** les deux premières (la trace d'un
projecteur de rang un vaut un ; le projecteur de paire est idempotent) et la quatrième (le
support double moins un, sur le registre). **Mesuré** : ces trois-là. Les valeurs de $O_1$ et
$O_3$ dépendent des tables du twirl — **citées**, pas rejouées.

In [5]:
# Pi = projecteur de paire a deux copies, rang un sur le registre de dimension 16.
Pi = np.outer(ket_omega, ket_omega)
rang_un = int(round(np.trace(Pi).real))
idempotent = np.allclose(Pi @ Pi, Pi)

lecture = lambda M: np.trace(M @ Pi).real / dim_R

l_un = lecture(np.eye(dim_R))     # <1> : la trace d'un projecteur de rang un, sur le registre
l_Pi = lecture(Pi)                # <Pi> : idempotent, donc tr(Pi^2) = tr(Pi)
l_O2_formule = (D_double - 1) / dim_R   # la quatrieme : support double moins un, sur le registre

print(f"rang de Pi = {rang_un}, idempotent : {idempotent}")
print(f"<1>  = {l_un:.6f}  = 1/16  ({abs(l_un - 1/16) < 1e-12})")
print(f"<Pi> = {l_Pi:.6f}  = 1/16  ({abs(l_Pi - 1/16) < 1e-12})")
print(f"<O2> (formule (D-1)/16) = {l_O2_formule:.6f}  = 3/16")
print("Cite : <O1> = 1/16, <O3> = 0 (tables du twirl, census de la source)")

rang de Pi = 1, idempotent : True
<1>  = 0.062500  = 1/16  (True)
<Pi> = 0.062500  = 1/16  (True)
<O2> (formule (D-1)/16) = 0.187500  = 3/16
Cite : <O1> = 1/16, <O3> = 0 (tables du twirl, census de la source)


### Lecture du résultat

Les trois vérifications à la main que la source propose sont des vérifications machine :
$\langle 1 \rangle = \mathrm{tr}(\Pi)/16 = 1/16$ parce que $\Pi$ est de rang un,
$\langle \Pi \rangle = \mathrm{tr}(\Pi^2)/16 = 1/16$ parce que $\Pi$ est idempotent, et
$\langle O_2 \rangle = (D-1)/16 = 3/16$ par l'arithmétique du support double. Les cinq lectures
sont **symétriques sous le swap** — pour les trois rejouées, c'est immédiat : l'état $\lvert\Omega\rangle$
est un vecteur propre exact de l'échange, donc toute lecture y est invariante.

## Équation 63 — le cycle impair à côté du swap

De la configuration debout, le **cycle d'échange** — un triangle de longueur trois — porte par la
parité de sa longueur le signe moins un ; il se tient **à côté** du signe de swap plus un et
n'empêche rien :

$$\sigma_{\text{cycle}} = (-1)^3 = -1, \qquad \sigma_{\text{swap}} = +1. \qquad (63)$$

**Mesuré** : les deux lignes — l'une est une parité, l'autre est la valeur propre mesurée plus haut.

In [6]:
longueur_cycle = 3                      # le triangle d'echange de la configuration debout
sigma_cycle = (-1) ** longueur_cycle
sigma_swap = sigma_amp                   # mesure a l'eq. 61

print(f"sigma_cycle = (-1)^{longueur_cycle} = {sigma_cycle:+d}")
print(f"sigma_swap  = {sigma_swap:+.15f}  (valeur propre mesuree de l'echange des copies)")
# flottant : la coexistence se teste a tolerance, pas par egalite exacte
print(f"les deux signes coexistent : "
      f"{sigma_cycle == -1 and abs(sigma_swap - 1.0) < 1e-12}")

sigma_cycle = (-1)^3 = -1
sigma_swap  = +1.000000000000000  (valeur propre mesuree de l'echange des copies)
les deux signes coexistent : True


### Lecture du résultat

Aucune tension n'est résolue ici, et le carnet ne doit pas en fabriquer : le $-1$ du cycle est un
**fait de parité** sur la longueur trois, le $+1$ du swap est une **valeur propre mesurée** sur le
ket de paire doublé. La source dit précisément cela — le signe du cycle « stands beside the swap
sign plus one and obstructs nothing ». Ce que le lecteur de la littérature appelle le signe
spin-statistiques ([76], correspondance par nom) n'est **pas** ce que ce chapitre établit : voir
la section Ouvert.

## Le plus petit objet du census — le porteur à cinq sites (figure 18)

Le census parcourt les dix-huit objets de la construction ; le plus petit est un **porteur de
cinq sites et cinq liens** : un qutrit $Q$ (dimension 3), un hub $H$ (dimension 4) et trois
qubits $q_1, q_2, q_3$. La figure 18 étiquette chaque site par sa dimension et chaque lien par
son type de paire :

| Lien | Type | Mesure du lien |
|---|---|---|
| $Q$–$H$ | $(3, 4)$ | $12$ |
| $q_1$–$H$, $q_2$–$H$, $q_3$–$H$ | $(2, 4)$ | $8$ chacun |
| $q_1$–$q_2$ | $(2, 2)$ | $4$ |

Sa configuration à deux copies **double chaque site et chaque paire**. Le census dit : le swap
**fixe les cinq paires**, le twirl d'étiquette **s'évanouit aux cinq**, les lectures sont
**symétriques aux cinq**, le signe de swap **plus un**. **Mesuré** : la structure (cinq sites,
dimensions, cinq liens, mesures) et le fait que la paire doublée de chaque lien est fixée par
l'échange des copies avec l'amplitude exactement plus un. Le twirl d'étiquette et les lectures
du census sont **cités** — leur construction appartient à la source.

In [7]:
# Le porteur de la figure 18 : cinq sites, cinq liens.
dims_porteur = {"Q": 3, "H": 4, "q1": 2, "q2": 2, "q3": 2}
liens_porteur = [("Q", "H"), ("q1", "H"), ("q2", "H"), ("q3", "H"), ("q1", "q2")]

mesures = [(a, b, dims_porteur[a] * dims_porteur[b]) for a, b in liens_porteur]
for a, b, m in mesures:
    print(f"lien {a:>2}-{b:<2} : type ({dims_porteur[a]},{dims_porteur[b]}), mesure {m}")
print(f"total : {len(dims_porteur)} sites, {len(liens_porteur)} liens, "
      f"somme des mesures {sum(m for _, _, m in mesures)}")

# La paire doublee de chaque lien : ket de paire v (x) v, l'echange des copies transpose les blocs.
# On prend pour v la paire maximalement intriquee sur la plus petite dimension du lien.
def ket_paire_max_entanglee(d_a, d_b):
    d = min(d_a, d_b)
    v = np.zeros(d_a * d_b)
    for i in range(d):
        v[i * d_b + i] = 1.0
    return v / np.sqrt(d)

print()
for a, b in liens_porteur:
    d_a, d_b = dims_porteur[a], dims_porteur[b]
    v = ket_paire_max_entanglee(d_a, d_b)
    fixee, amplitude = paire_double_fixee(v)
    print(f"paire doublee {a:>2}-{b:<2} : fixee par l'echange = {fixee}, amplitude {amplitude:+.6f}")
    assert fixee and abs(amplitude - 1.0) < 1e-12

lien  Q-H  : type (3,4), mesure 12
lien q1-H  : type (2,4), mesure 8
lien q2-H  : type (2,4), mesure 8
lien q3-H  : type (2,4), mesure 8
lien q1-q2 : type (2,2), mesure 4
total : 5 sites, 5 liens, somme des mesures 40

paire doublee  Q-H  : fixee par l'echange = True, amplitude +1.000000
paire doublee q1-H  : fixee par l'echange = True, amplitude +1.000000
paire doublee q2-H  : fixee par l'echange = True, amplitude +1.000000
paire doublee q3-H  : fixee par l'echange = True, amplitude +1.000000
paire doublee q1-q2 : fixee par l'echange = True, amplitude +1.000000


### Lecture du résultat

Les cinq paires doublées sont fixées par l'échange des copies, à l'amplitude $+1$ exactement.
Le fait est **structural** : une paire doublée $v \otimes v$ est échangée en $v \otimes v$ — la
matrice $v v^{\mathsf T}$ est symétrique — et c'est pourquoi le census peut le constater aux
dix-huit objets sans exception. Ce que le carnet mesure ici est la **face amplitude** du porteur ;
le twirl d'étiquette et les lectures symétriques du census restent cités, parce que leur
construction (le groupe d'identification, ses tables) appartient à la source.

## Cité — le census jamais rejoué

Tout ce qui suit est **cité depuis la source**, jamais recalculé par ce carnet :

- **Dix-huit objets** énumérés (treize corps, cinq porteurs), repris dans l'ordre contre leurs
  tables : à chacun, signe de swap **plus un à la face amplitude**, lectures symétriques à
  chaque paire, signe de cycle **moins un à chaque objet**.
- Le **swap de doublet** — l'échange des objets bidimensionnels d'un corps — se réalise par des
  transpositions aux dix-huit objets, de signe $(-1)$ porté au compte de doublets : **neuf objets
  à plus un, neuf à moins un**, « en accord avec les noms de statistiques du Standard Model » ;
  c'est un théorème conditionnel, *second fallback* des résultats de statistiques ouverts.
- **Exclusion** : l'exclusion de la paire $d$ plus première est un corollaire du comptage aux
  **1691 liens** de la construction sous les deux lectures de base ; **no-cloning** ([25],
  correspondance par nom) est le même corollaire à une paire — la copie exclue par le compte aux
  **1229 des 1334 sites**, par le lavage aux **105 restants**.
- **Contre-tests et contrôles** : le premier déplace **346 cellules** (dont les cellules de
  lavage et de symétrie au type deux-qubits), le second **506** (dont la première défaillance) ;
  cinq contre-tests aux lettres épinglées, onze contrôles lus identiques aux deux moteurs ;
  reproduction par un **second moteur** construit à part.
- Le paquet vérifié par machine : « 18 objects enumerated », certifié au dépôt — il prouve les
  énoncés de ce chapitre sous les définitions du compendium et **aucun théorème de la
  littérature**.
- Correspondances : [76] théorème spin-statistiques, [25] no-cloning, [33] échange comme
  permutation — toutes **par nom**.

## Ouvert — ligne à ligne

Ce que la source **ne prouve pas**, tenu pour ce qu'il est :

1. **Le signe spin-statistiques de la littérature n'est pas obtenu.** Le swap porte $+1$ à la
   face amplitude à chaque objet ; le $-1$ vit à la face permutation et au cycle impair, « à
   côté, n'empêchant rien ». La correspondance avec le théorème spin-statistiques [76] est
   **par nom** — le chapitre n'établit pas le lien quantique spin $\leftrightarrow$ statistique.
2. **Premier ouvert : la biconditionnelle « cycle impair si et seulement si un doublet ».** Le
   cycle debout est impair à chaque objet, le compte de doublets vaut un aux neuf corps de
   chaîne seuls — la clause **part à neuf des dix-huit objets** (le mot $W$ en premier).
3. **Deuxième ouvert : l'admissibilité du swap de doublet comme identification.** Il porte
   $-1$ au compte de doublets à chaque objet et est admissible aux lectures lisibles propres de
   **quinze objets** — elle part aux **trois autres** (le mot gluon en premier).
4. **Prémices ouvertes citées** : résultats ouverts 9.8, 18.5, 18.6, 28.3, 28.4, 241.1, 241.2,
   241.5 à 241.7 — la chaîne de prémisses remonte à l'axiome sur douze chapitres, la plus longue
   du compendium.
5. **Standing : un théorème conditionnel** — conditionnel sur le texte de l'axiome et sur chaque
   prémisse à son propre statut. **Preprint non relu.**

## Ce que ce carnet ne prouve pas, non plus

Les symétries mesurées ici (éq. 61-63, porteur) sont les cas élémentaires que la source invite
à reprendre ; le census aux dix-huit objets, le lavage par twirl et les tables $O_1, O_2, O_3$
ne sont ni recalculés ni approchés — les citer n'est pas les vérifier.

## Croisements

- **Chaînes-02** : les lettres de Weyl $I/X/Z/XZ$ du swap d'intrication — le sous-produit du
  lavage à chaque qudit partagé — sont les mêmes qu'au chapitre 40 ; la relation de Weyl y était
  déjà mesurée aux dimensions 2-3-4.
- **Chaînes-03** : le lavage à un acte du chapitre 42 est le geste qui rend le swap « lavé par
  le twirl » ; le porteur à cinq sites est à l'indiscernabilité ce que le corps à deux docks
  était à la mesure — le plus petit objet du census.
- **Chaînes-05 (candidat)** : le chapitre 44 — l'énergie comme opérateur à première puissance
  et le pas (Hamiltonien, éq. 64-67), prémices 18.5-18.6 et 241.x partagées.

## Références

Source : compendium Chain Theory (v0.1 working draft, 754 p.), chapitre 43, p. 119-121 — lu
depuis la copie archivée au gisement avant écriture, jamais commise. Correspondances citées par
la source : [25] no-cloning, [33] échange comme permutation, [76] théorème spin-statistiques.

## Exercices

### Exercice 1 — le signe à $d = 5$

La formule $d(d-1)/2$ donne le nombre de transpositions de la permutation d'étiquettes. Pour
$d = 5$ : combien de transpositions, quel signe, combien d'étiquettes fixes ? Le tableau du
carnet s'arrête à $d = 4$ — prolonge-le.

# Etape 1 : appeler `signe_permutation_etiquettes(5)` et comparer le compte a la formule.
# Etape 2 : verifier que le signe vaut (-1)**(d*(d-1)//2).

In [8]:
# Exercice 1 : prolonger le tableau des signes a d = 5.
# Attendu : le compte de transpositions, le signe, le nombre d'etiquettes fixes.
result = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


### Exercice 2 — la ligne (61) pour une paire de qutrits

Refais la ligne (61) au type deux-qutrits : deux copies d'un site de dimension 3, la paire à
deux copies $\lvert \omega_3 \rangle \otimes \lvert \omega_3 \rangle$. Quels $r$, $D$,
$\dim R$, rang de Schmidt à travers les **sites doublés** $(A_1 A_2)|(B_1 B_2)$, et
$\sigma_{\text{amp}}$ ? Souviens-toi de la bipartition : à travers les copies, le rang vaut un.

# Etape 1 : construire ket_omega_3 (dimension 9 par copie, 81 au total).
# Etape 2 : mesurer le rang de Schmidt a travers les sites doubles et le signe d'amplitude,
#           comme a l'eq. 61 (transpose (0, 2, 1, 3) avant le reshape).

In [9]:
# Exercice 2 : la ligne (61) au type deux-qutrits (d = 3).
# Attendu : r, D, dim R, rang de Schmidt, sigma_amp.
result = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


### Exercice 3 — la lecture $\langle O_2 \rangle$ à un autre type

La quatrième lecture vaut $(D-1)/\dim R$ — le support double moins un, sur le registre. Au type
deux-qutrits de l'exercice 2 ($D = 6$ après doublement des sites de dimension 3, registre de
dimension 81... vérifie ces deux valeurs avant de conclure) : que vaut $\langle O_2 \rangle$ ?
Le résultat est-il encore un rationnel exact à dénominateur 81 ?

# Etape 1 : poser D et dim R pour le type deux-qutrits (attention : D est le support double).
# Etape 2 : evaluer (D - 1) / dim R et dire si la fraction se simplifie.

In [10]:
# Exercice 3 : la lecture <O2> au type deux-qutrits.
# Attendu : les valeurs de D et dim R, puis (D - 1) / dim R.
result = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


## À retenir

- Le signe de l'échange dépend de la **face** où on le lit : $+1$ à la face amplitude (mesuré,
  vecteur propre exact du ket de paire doublé), $-1$ à la face permutation (parité de $d(d-1)/2$
  transpositions des étiquettes produit), $-1$ au cycle impair de longueur trois (parité de la
  longueur).
- La ligne (61) au type deux-qubits — $r = 2$, $D = 4$, $\dim R = 16$, rang de Schmidt 4 — est
  entièrement rejouée ; les lectures (62) que la source invite à reprendre à la main le sont
  aussi ($1/16$, $1/16$, $3/16$) ; le porteur à cinq sites de la figure 18 est construit et ses
  cinq paires doublées sont fixées à l'amplitude $+1$.
- Le census aux dix-huit objets, le twirl d'identification, les tables $O_1, O_2, O_3$, les
  comptes 1691/1334/1229/105 et les correspondances spin-statistiques/no-cloning sont **cités**,
  jamais rejoués — et le chapitre lui-même est un **théorème conditionnel** sur un preprint non
  relu : le statut épistémique est posé avant le calcul, comme partout dans cette série.

## Vérifiez votre compréhension

1. Le même état donne un rang de Schmidt de 4 à travers $(A_1A_2)|(B_1B_2)$ et de 1 à travers la
   coupure des copies. Pourquoi ?
2. Que valent $\sigma_{\text{amp}}$ et $\sigma_{\text{perm}}$, et comment les deux signes
   coexistent-ils ?
3. Quelles probabilités le carnet retrouve-t-il sur les cinq lectures de l'équation 62 ?
4. Pourquoi le cycle impair de longueur trois porte-t-il le signe moins ?

<details>
<summary>Réponses</summary>

1. Parce que la bipartition fait le rang. À travers les sites doublés, deux copies d'un état de
   rang $r = 2$ donnent $r^2 = 4$ ; à travers la coupure des copies, l'état est un **produit**
   des deux paires, donc de rang 1. La source mesure à la coupure des sites.
2. $\sigma_{\text{amp}} = +1$ (le ket de paire doublé est vecteur propre exact de l'échange des
   copies) et $\sigma_{\text{perm}} = -1$ (parité de $d(d-1)/2 = 1$ transposition sur les quatre
   étiquettes produit à $d = 2$). Ce ne sont pas deux réponses à la même question : le signe
   dépend de la **face** à laquelle on le lit.
3. $1/16$, $1/16$ et $3/16$, sur $\dim R = 16$ — les lectures que la source invite à reprendre à
   la main.
4. Par la parité de sa longueur : trois est impair, donc $-1$.

</details>


## Pour aller plus loin

[Chaînes-05 — l'énergie, l'hamiltonien et la première puissance](Chaines-05-Energie-Hamiltonien-Premiere-Puissance-Python.ipynb)
prend la suite du fil avec le chapitre 44 du compendium : l'énergie comme opérateur de première
puissance et les équations 64 à 67. La discipline de lecture y est la même — ce que la source
dérive, ce qu'elle suppose, et ce que le carnet rejoue.

En amont, [Chaînes-03 — la mesure : le lavage à un acte et la POVM](Chaines-03-Mesure-Lavage-POVM-Python.ipynb)
fournit le cadre de lecture collective dont ce carnet tire le signe, et [Chaînes-02](Chaines-02-Born-Unitarite-Incertitude-Python.ipynb)
y construit les lettres de Weyl.

Hors de la série, [Jauge-01 — le théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb)
offre le contraste utile : les lois de conservation y sont des théorèmes non conditionnels, là où
le signe spin-statistique reste ici une conséquence d'un théorème conditionnel sur un preprint non
relu. Le chapitre lui-même est cité dans la section « Cité — le census jamais rejoué » : c'est le
point de reprise pour qui veut vérifier les comptes 1691/1334/1229/105.
